# Quality Assessment Method Comparison

Compare image quality assessment methods on PhotoTriage dataset:
- **Performance metrics**: Top-1/2 Accuracy, MRR, Runtime
- **Correlation analysis**: Which methods agree on series difficulty?
- **Method wins**: Cases where one method succeeds but others fail
- **Failure analysis**: Patterns in failure cases

## Configuration Options

**Option 1 - Single folder**: Specify exact benchmark directory
```python
AUTO_SCAN = False
BENCHMARK_DIR = "outputs/quality_benchmarks/benchmark_2025-11-14_00-06-59"
```

**Option 2 - Auto-scan** (recommended): Automatically find latest benchmark
```python
AUTO_SCAN = True
BENCHMARK_BASE_DIR = "outputs/quality_benchmarks"
```



In [ ]:
# === Imports ===
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Import from specific analysis modules (not from __init__ since it's empty)
from sim_bench.quality_assessment.analysis.load_results import (
    load_quality_results,
    merge_per_series_metrics,
)
from sim_bench.quality_assessment.analysis.correlation import (
    compute_correlation_matrix,
)
from sim_bench.quality_assessment.analysis.visualization import (
    plot_performance_comparison,
    plot_runtime_comparison,
    plot_efficiency_comparison,
    plot_correlation_heatmap,
)
from sim_bench.quality_assessment.analysis.method_wins import (
    find_method_wins,
    visualize_method_wins,
    get_top_methods_by_accuracy,
)
from sim_bench.quality_assessment.analysis.failure_analysis import (
    analyze_failures,
    plot_failure_analysis,
)
from sim_bench.analysis.export import export_notebook_to_pdf, archive_notebook
from sim_bench.analysis.utils import get_project_root

sns.set_style('whitegrid')
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

print("✓ Imports successful")

## Configuration



In [ ]:
# === Configuration ===
PROJECT_ROOT = get_project_root()
EXCLUDE_METHODS_REGEX = 'composite'
# Choose scanning mode
AUTO_SCAN = True  # Set to False to use specific folder

if AUTO_SCAN:
    # Option 1: Auto-scan from base directory (uses latest)
    BENCHMARK_BASE_DIR = PROJECT_ROOT / "outputs" / "pairwise_benchmark_3hour"
    BENCHMARK_DIR = None
    USE_LATEST = False
else:
    # Option 2: Specify exact benchmark directory
    BENCHMARK_BASE_DIR = None
    BENCHMARK_DIR = PROJECT_ROOT / "outputs" / "quality_benchmarks" / "benchmark_2025-11-14_00-06-59"
    USE_LATEST = False

# Analysis parameters
TOP_N_METHODS = 10  # Number of top methods to show in wins visualization
TOP_N_IMAGES = 3   # Number of top images to show per method

# Export options
EXPORT_HTML = True  # Export notebook to HTML at the end

print(f"Scanning mode: {'AUTO' if AUTO_SCAN else 'MANUAL'}")
if AUTO_SCAN:
    print(f"Base directory: {BENCHMARK_BASE_DIR.relative_to(PROJECT_ROOT)}")
else:
    print(f"Benchmark directory: {BENCHMARK_DIR.relative_to(PROJECT_ROOT)}")



## 1. Load Data



In [ ]:
# Load benchmark results
methods_summary_df, detailed_results_df, series_data_dict, benchmark_dir = load_quality_results(
    benchmark_dir=BENCHMARK_DIR,
    auto_scan=AUTO_SCAN,
    base_dir=BENCHMARK_BASE_DIR,
    use_latest=USE_LATEST
)

print(f"\n{'='*80}")
print("DATA SUMMARY")
print(f"{'='*80}")
print(f"Benchmark directory: {benchmark_dir.name}")
print(f"Methods found: {len(methods_summary_df)}")
print(f"Methods: {', '.join(methods_summary_df['method'].tolist())}")
print(f"\nMethods Summary:")
display(methods_summary_df)

print(f"\nDetailed Results:")
display(detailed_results_df)



## 2. Performance Metrics Comparison



In [ ]:
# Plot performance comparison
fig = plot_performance_comparison(methods_summary_df)
plt.show()



In [ ]:
# Plot runtime comparison
fig = plot_runtime_comparison(detailed_results_df)
plt.show()



In [ ]:
# Plot efficiency (accuracy vs speed)
fig = plot_efficiency_comparison(methods_summary_df)
plt.show()



## 3. Correlation Matrix

Analyze which methods agree on series difficulty (Top-1 Accuracy correlation).



In [ ]:
# # Merge per-series data for correlation analysis
# methods = methods_summary_df['method'].tolist()
# merged_df = merge_per_series_metrics(series_data_dict, methods=methods, metric='correct')

# print(f"Merged {len(merged_df)} series across {len(methods)} methods")

# # Compute correlation matrix
# corr_matrix = compute_correlation_matrix(merged_df, methods=methods, metric='correct')

# print(f"\nCorrelation Matrix:")
# display(corr_matrix)

# # Plot correlation heatmap
# fig = plot_correlation_heatmap(corr_matrix, methods=methods)
# plt.show()



In [ ]:
# Per-method visualizations
for method in ['Combined-RuleBased', 'Sharpness', 'Colorfulness']:
    print(f"\n{'='*80}")
    print(f"{method} - Performance by Attribute")
    print(f"{'='*80}")
    fig = plot_method_performance_by_label(label_presence, method, figsize=(14, 6))
    if fig:
        plt.show()

In [ ]:
# Composition deep-dive
composition_pairs = combined_df[combined_df['label_composition'] > 0]
print(f"\nCOMPOSITION ANALYSIS ({len(composition_pairs)} pairs where 'composition' mentioned)")
print("="*80)

for method in pairwise_methods:
    correct_col = f'{method}_correct'
    comp_acc = composition_pairs[correct_col].mean()
    overall_acc = combined_df[correct_col].mean()
    diff = comp_acc - overall_acc
    sign = "✓" if diff > 0 else "✗" if diff < 0 else "="
    print(f"{sign} {method:25s}: {comp_acc:.3f} (overall: {overall_acc:.3f}, diff: {diff:+.3f})")

In [ ]:
# Performance when specific attributes mentioned
label_cols = [col for col in combined_df.columns if col.startswith('label_') and 
              col not in ['label_no_reason_given', 'label_no_specific_attribute']]
label_presence = compute_accuracy_by_label_presence(combined_df, pairwise_methods, label_cols)

print("\nAccuracy when attribute PRESENT vs ABSENT:")
print("="*80)
display(label_presence.sort_values('present_count', ascending=False))

In [ ]:
# Accuracy by majority label (user's main reason for choice)
majority_label_accuracy = compute_accuracy_by_attribute(combined_df, pairwise_methods, 'majority_label')

print("Method Accuracy by User-Labeled Attribute:")
print("="*80)
display(majority_label_accuracy.sort_values('count', ascending=False))

# Visualize heatmap
fig = plot_accuracy_by_majority_label(majority_label_accuracy, pairwise_methods, figsize=(14, 8))
plt.show()

In [ ]:
# Import pairwise utilities
sys.path.insert(0, str(Path.cwd()))
from merge_pairwise_results import load_pairwise_results, merge_pairwise_results
from analyze_pairwise_by_attributes import (
    merge_results_with_labels,
    compute_accuracy_by_attribute,
    compute_accuracy_by_label_presence,
    plot_accuracy_by_majority_label,
    plot_method_performance_by_label,
)

# Load merged results and user labels
MERGED_RESULTS_PATH = benchmark_dir / "merged_pairwise_results.csv"
LABELS_PATH = Path(r"D:\Similar Images\automatic_triage_photo_series\photo_triage_pairs_keyword_labels.csv")

pairwise_merged = pd.read_csv(MERGED_RESULTS_PATH)
combined_df = merge_results_with_labels(MERGED_RESULTS_PATH, LABELS_PATH)
pairwise_methods = [col.replace('_correct', '') for col in combined_df.columns if col.endswith('_correct')]

print(f"Loaded {len(combined_df)} pairs with labels ({len(combined_df)/len(pairwise_merged)*100:.1f}% coverage)")
print(f"Methods: {', '.join(pairwise_methods)}")

---
# Part 2: Pairwise Analysis with User Attributes

Compare method accuracy across different user-labeled quality attributes.